In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/janadarwish/transaction/transactions-csv-(working-link).csv


In [2]:
import pandas as pd
import numpy as np
import shutil
from pathlib import Path

In [3]:
src = '/kaggle/input/datasets/janadarwish/transaction/transactions-csv-(working-link).csv'
print("Found:", src)

Found: /kaggle/input/datasets/janadarwish/transaction/transactions-csv-(working-link).csv


In [4]:
Path('data').mkdir(parents=True, exist_ok=True)
Path('output').mkdir(parents=True, exist_ok=True)
shutil.copy(src, 'data/transactions.csv')

'data/transactions.csv'

In [5]:
df = pd.read_csv('data/transactions.csv')
df.head()

,transaction_id,date,client_id,region,amount
0,TXN-58392,2023-08-01,APX-4921,East,342.15
1,TXN-58393,2023-08-01,APX-9284,North,6241.90
2,TXN-58394,2023-08-02,APX-1120,South,819.50
3,TXN-58395,2023-08-02,APX-8472,West,9201.25
4,TXN-58396,2023-08-03,APX-4921,East,5100.00


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   transaction_id  30 non-null     object 
 1   date            30 non-null     object 
 2   client_id       30 non-null     object 
 3   region          30 non-null     object 
 4   amount          30 non-null     float64
dtypes: float64(1), object(4)
memory usage: 1.3+ KB


In [7]:
df['amount'] = pd.to_numeric(df['amount'].astype(str).str.replace(r'[$,]', '', regex=True))

In [8]:
df['transaction_category'] = np.where(df['amount'] > 5000, 'High', 'Standard')
df['scaled_amount'] = df['amount'] * 0.01
df.head()

,transaction_id,date,client_id,region,amount,transaction_category,scaled_amount
0,TXN-58392,2023-08-01,APX-4921,East,342.15,Standard,3.4215
1,TXN-58393,2023-08-01,APX-9284,North,6241.90,High,62.4190
2,TXN-58394,2023-08-02,APX-1120,South,819.50,Standard,8.1950
3,TXN-58395,2023-08-02,APX-8472,West,9201.25,High,92.0125
4,TXN-58396,2023-08-03,APX-4921,East,5100.00,High,51.0000


In [9]:
summary = df.groupby('region').agg({'amount': ['sum', 'mean']})
summary.to_csv('output/executive_summary.csv')
summary

amount             
             sum         mean
region                       
East    21434.50  2679.312500
North   30081.75  3760.218750
South   24063.69  3437.670000
West    34187.80  4883.971429

In [10]:
print(open('output/executive_summary.csv').read())

,amount,amount
,sum,mean
region,,
East,21434.5,2679.3125
North,30081.75,3760.21875
South,24063.69,3437.6699999999996
West,34187.8,4883.971428571429



In [11]:
%%writefile output/reply_draft.txt
Subject: RE: URGENT: Apex Financial Summary Pipeline Rewrite

Hi Dr. Diop,

Attached are pipeline.ipynb and executive_summary.csv for the Apex Financial review.

The pipeline uses np.where() for the High/Standard flag, broadcasting (df['amount'] * 0.01) for scaling, and a single groupby().agg() for the regional summary. Every operation runs over whole columns in compiled code instead of looping row by row, which removes the per-row Python overhead behind the timeouts.

Best,
[Your name]

Writing output/reply_draft.txt


> ***Vectorization** means applying an operation to an entire array or column at once using optimized, compiled C code in NumPy/Pandas instead of processing one record at a time. It performs better than a Python for-loop because it avoids the interpreter overhead of running Python code and type checks for every row, and it works on contiguous memory.*

In [12]:
!ls -R /kaggle/working

/kaggle/working:
data  __notebook__.ipynb  output

/kaggle/working/data:
transactions.csv

/kaggle/working/output:
executive_summary.csv  reply_draft.txt
